In [ ]:
# 1. Desinstalar la versión estándar por si acaso (para evitar conflictos)
!pip uninstall -y ultralytics

# 2. Instalar la versión ESPECÍFICA de YOLOv12 (igual que en el entrenamiento)
!pip install git+https://github.com/sunsmarterjie/yolov12.git
!pip install fpdf

# 3. Importar y cargar
import torch
from ultralytics import YOLO

# 3. Instalamos el menú de la App
!pip install streamlit-option-menu

  Cloning https://github.com/sunsmarterjie/yolov12.git to /tmp/pip-req-build-23qx1uc6
  Running command git clone --filter=blob:none --quiet https://github.com/sunsmarterjie/yolov12.git /tmp/pip-req-build-23qx1uc6
  Resolved https://github.com/sunsmarterjie/yolov12.git to commit d3cbe103b992ff6dd20c722e84e1b6f18f4c5efc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.63-py3-none-any.whl size=910521 sha256=c563c9f7d4d109f6f31d6328f1134e164f980c13a5ed6199015db28d90ae06cb
  Stored in directory: /tmp/pip-ephem-wheel-cache-5_11yir1/wheels/51/14/51/9f3f73766e89100fb390b031d2290899b2fef7b57d9a74c6dc
Successfully built ultralytics
  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=eb38f13820cf631a9a953d2e2d7e0c656bc5d2bb1a417ebc7b95c28936a0a1be
  Stored in directory: /root/.cach

In [ ]:
import shutil
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 1. Copiar modelos desde Drive
modelos = ['best.pt', 'best_v8_sard.pt', 'best_yolov12.pt']
for m in modelos:
    src = f'/content/drive/MyDrive/TFG/{m}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{m}')

# 2. Recuperar el historial de estadísticas correcto
csv_name = 'registro_analisis_v2.csv'
path_drive = f'/content/drive/MyDrive/TFG/{csv_name}'

if os.path.exists(path_drive):
    shutil.copy(path_drive, f'/content/{csv_name}')
    print(f" Archivo '{csv_name}' recuperado de Drive.")
else:
    print(f" No se encontró '{csv_name}' en Drive, se creará uno nuevo al analizar.")

Mounted at /content/drive


In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import tempfile
import os
import time
import uuid
import pandas as pd
import psutil
import subprocess
import gc
from datetime import datetime
from ultralytics import YOLO
from PIL import Image
from streamlit_option_menu import option_menu
import altair as alt
from fpdf import FPDF

# --- 1. CONFIGURACIÓN E INTERFAZ ---
st.set_page_config(page_title="Sistema SAR - TFG", layout="wide")

st.markdown("""
<style>
    .stApp { background-color: #0E1117; }
    h1, h2, h3, p, label { color: #E0E0E0 !important; font-family: 'Segoe UI', sans-serif; }
    div[data-testid="stMetric"] {
        background-color: #262730; border: 1px solid #41444C; padding: 15px; border-radius: 8px;
    }
    .stButton>button { background-color: #262730; color: white; border: 1px solid #41444C; width: 100%; }
    .stButton>button:hover { background-color: #D84315; border-color: #FF5722; color: white; }

    div[data-baseweb="select"] > div {
        background-color: #262730; color: white; border-color: #41444C;
    }
</style>
""", unsafe_allow_html=True)

# --- 2. GESTIÓN DE ESTADO ---
if 'source_video_path' not in st.session_state: st.session_state['source_video_path'] = None
if 'source_video_name' not in st.session_state: st.session_state['source_video_name'] = None
if 'raw_video_path' not in st.session_state: st.session_state['raw_video_path'] = None
if 'final_video_path' not in st.session_state: st.session_state['final_video_path'] = None
if 'last_quality' not in st.session_state: st.session_state['last_quality'] = None
if 'source_image_path' not in st.session_state: st.session_state['source_image_path'] = None

# --- 3. FUNCIONES ---
@st.cache_resource(show_spinner="Cargando IA...")
def load_model_sar(path):
    possible_paths = [path, f"/content/{path}", f"/content/drive/MyDrive/TFG/{path}"]
    for p in possible_paths:
        if os.path.exists(p): return YOLO(p), "CUSTOM"
    return YOLO('yolov8n.pt'), "BASE"

DRIVE_PATH = '/content/drive/MyDrive/TFG'
CSV_NAME = 'registro_analisis_v2.csv'
LOG_FILE = os.path.join(DRIVE_PATH, CSV_NAME) if os.path.exists(DRIVE_PATH) else CSV_NAME

def save_log(tipo, duracion, fps, modelo, cpu):
    nueva_fila = {
        'ID': str(uuid.uuid4())[:8], 'Fecha': datetime.now().strftime("%Y-%m-%d"),
        'Hora': datetime.now().strftime("%H:%M:%S"), 'Tipo': tipo, 'Modelo': modelo,
        'Duracion_Sec': round(float(duracion), 2), 'FPS_Promedio': round(float(fps), 1),
        'CPU_Usage': round(float(cpu), 1)
    }
    try:
        df = pd.read_csv(LOG_FILE) if os.path.exists(LOG_FILE) else pd.DataFrame()
        df = pd.concat([df, pd.DataFrame([nueva_fila])], ignore_index=True)
        df.to_csv(LOG_FILE, index=False)
    except: pass

def create_pdf_report(df):
    class PDF(FPDF):
        def header(self):
            self.set_font('Arial', 'B', 15)
            self.cell(0, 10, 'Informe Tecnico de Mision SAR', 0, 1, 'C')
            self.set_font('Arial', 'I', 10)
            self.cell(0, 10, f'Generado: {datetime.now().strftime("%Y-%m-%d %H:%M")}', 0, 1, 'C')
            self.ln(5)

        def footer(self):
            self.set_y(-15)
            self.set_font('Arial', 'I', 8)
            self.cell(0, 10, f'Pagina {self.page_no()}', 0, 0, 'C')

    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    # Resumen Ejecutivo
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, "1. Resumen Ejecutivo", 0, 1)
    pdf.set_font("Arial", size=11)

    total_ops = len(df)
    avg_fps = df['FPS_Promedio'].mean()
    fav_model = df['Modelo'].mode()[0] if not df.empty else "N/A"

    pdf.cell(0, 8, f"Total de Operaciones Realizadas: {total_ops}", 0, 1)
    pdf.cell(0, 8, f"Rendimiento Medio Global: {avg_fps:.2f} FPS", 0, 1)
    pdf.cell(0, 8, f"Modelo de IA Predominante: {fav_model}", 0, 1)
    pdf.ln(10)

    # Tabla de Datos
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, "2. Registro de Actividad (Ultimos 15)", 0, 1)
    pdf.set_font("Courier", size=9) # Fuente monoespaciada para tabla simple

    # Cabecera Tabla
    header = f"{'FECHA':<12} {'HORA':<10} {'TIPO':<8} {'MODELO':<10} {'FPS':<6} {'DUR(s)':<8}"
    pdf.cell(0, 8, header, 0, 1)
    pdf.line(10, pdf.get_y(), 190, pdf.get_y())

    # Filas
    for index, row in df.head(15).iterrows():
        line = f"{str(row['Fecha']):<12} {str(row['Hora']):<10} {str(row['Tipo']):<8} {str(row['Modelo']):<10} {str(row['FPS_Promedio']):<6} {str(row['Duracion_Sec']):<8}"
        pdf.cell(0, 6, line, 0, 1)

    return pdf.output(dest='S').encode('latin-1', 'replace')

def transcode_video(input_path, mode):
    suffix = mode.split()[0].replace("(", "").replace(")", "")
    output_path = input_path.replace(".mp4", f"_web_{suffix}.mp4")

    common_flags = ["-movflags", "+faststart", "-g", "30", "-threads", "4"]

    if mode == "Alta Calidad (HD)":
        params = ["-preset", "fast", "-crf", "23", "-maxrate", "4M", "-bufsize", "8M"]
        vf = []
    elif mode == "Baja Latencia (SD)":
        params = ["-preset", "ultrafast", "-crf", "32", "-maxrate", "1M", "-bufsize", "2M"]
        vf = ["-vf", "scale=-2:480"]
    else: # AUTO
        params = ["-preset", "ultrafast", "-crf", "26", "-maxrate", "2.5M", "-bufsize", "5M"]
        vf = []

    cmd = ["ffmpeg", "-y", "-i", input_path, "-vcodec", "libx264", "-pix_fmt", "yuv420p"] + \
          params + vf + common_flags + [output_path]

    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path

# --- 4. BARRA LATERAL ---
with st.sidebar:
    st.title("⚙️ Configuración")
    model_option = st.radio("IA Seleccionada:", ["YOLO v5", "YOLO v8", "YOLO v12"], index=2)
    paths = {"YOLO v5": "best.pt", "YOLO v8": "best_v8_sard.pt", "YOLO v12": "best_yolov12.pt"}
    model, status = load_model_sar(paths[model_option])
    if "Error" in str(status): st.error(status)
    else: st.success(f"✓ {model_option} Listo")
    conf_threshold = st.slider("Confianza", 0.0, 1.0, 0.35, 0.05)

# --- 5. PANEL PRINCIPAL ---
st.title("🛰️ Plataforma SAR: Localización de Personas")
selected = option_menu(None, ["Análisis Video", "Análisis Imagen", "Panel Estadístico"],
    icons=["camera-video", "image", "bar-chart-line"], orientation="horizontal")

if selected == "Análisis Video":
    uploaded_video = st.file_uploader("Cargar Video", type=['mp4', 'avi', 'mov'])

    if uploaded_video is not None:
        if st.session_state['source_video_name'] != uploaded_video.name:
            tfile = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
            tfile.write(uploaded_video.read())
            st.session_state['source_video_path'] = tfile.name
            st.session_state['source_video_name'] = uploaded_video.name
            st.session_state['raw_video_path'] = None
            st.session_state['final_video_path'] = None
            st.session_state['last_quality'] = None
            gc.collect()

    if st.session_state['source_video_path'] and os.path.exists(st.session_state['source_video_path']):
        c_info, c_btn = st.columns([3, 1])
        with c_info: st.info(f"📂 Archivo Activo: **{st.session_state['source_video_name']}**")
        with c_btn:
            if st.button("❌ Quitar"):
                for k in list(st.session_state.keys()): st.session_state[k] = None
                st.rerun()

        if st.button("▶ Iniciar Análisis IA"):
            cap = cv2.VideoCapture(st.session_state['source_video_path'])
            fps_in = cap.get(cv2.CAP_PROP_FPS)
            w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            raw_path = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4').name
            out = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (w, h))

            prog = st.progress(0)
            status_txt = st.empty()
            status_txt.text(f"🚀 Analizando {total_frames} frames...")

            start_t = time.time()
            cpu_track = []
            processed = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break

                # --- VUELTA A PREDICT (SIN TRACKING ID) ---
                res = model.predict(frame, conf=conf_threshold, verbose=False)
                out.write(res[0].plot())

                processed += 1
                if processed % 10 == 0: prog.progress(min(processed/total_frames, 1.0))
                if processed % 30 == 0: cpu_track.append(psutil.cpu_percent())

            cap.release()
            out.release()

            dur = time.time() - start_t
            avg_cpu = sum(cpu_track)/len(cpu_track) if cpu_track else 0
            save_log("VIDEO", dur, processed/dur if dur>0 else 0, model_option, avg_cpu)

            st.session_state['raw_video_path'] = raw_path
            st.session_state['final_video_path'] = None
            st.session_state['last_quality'] = None

            status_txt.success("✅ Análisis IA Completado.")
            gc.collect()

        if st.session_state['raw_video_path'] and os.path.exists(st.session_state['raw_video_path']):
            st.divider()
            col_r1, col_r2 = st.columns([2, 1])
            with col_r1: st.markdown("### 🎬 Resultado Final")
            with col_r2:
                quality = st.selectbox(
                    "Calidad Streaming:",
                    ["Auto (Prioridad Fluidez)", "Alta Calidad (HD)", "Baja Latencia (SD)"],
                    key="quality_selector"
                )

            video_to_show = None
            if (st.session_state['final_video_path'] is not None and
                os.path.exists(st.session_state['final_video_path']) and
                st.session_state['last_quality'] == quality):
                video_to_show = st.session_state['final_video_path']
                st.caption("⚡ Cargado desde caché (Instantáneo)")
            else:
                with st.spinner(f"⚡ Optimizando buffer para {quality}..."):
                    video_to_show = transcode_video(st.session_state['raw_video_path'], quality)
                    st.session_state['final_video_path'] = video_to_show
                    st.session_state['last_quality'] = quality

            st.video(video_to_show, format="video/mp4")

elif selected == "Análisis Imagen":
    up_img = st.file_uploader("Subir Imagen", type=['jpg', 'png'])
    if up_img:
        tfile_img = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
        tfile_img.write(up_img.read())
        st.session_state['source_image_path'] = tfile_img.name

    if st.session_state['source_image_path'] and os.path.exists(st.session_state['source_image_path']):
        st.info("🖼️ Imagen cargada.")
        if st.button("Procesar Imagen"):
            img = Image.open(st.session_state['source_image_path'])
            start_t = time.time()
            cpu_val = psutil.cpu_percent()
            res = model.predict(img, conf=conf_threshold)
            st.image(res[0].plot(), caption="Resultado", use_container_width=True)
            save_log("IMAGEN", time.time() - start_t, 0, model_option, cpu_val)
            gc.collect()
        if st.button("❌ Quitar Imagen"):
            st.session_state['source_image_path'] = None
            st.rerun()

elif selected == "Panel Estadístico":
    if os.path.exists(LOG_FILE):
        try:
            df = pd.read_csv(LOG_FILE)
            cols = ['Duracion_Sec', 'FPS_Promedio', 'CPU_Usage']
            for c in cols:
                if c not in df.columns: df[c] = 0.0
                df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

            # Preparar datos secuenciales
            df['Fecha_Hora'] = pd.to_datetime(df['Fecha'].astype(str) + ' ' + df['Hora'].astype(str))
            df = df.sort_values(by='Fecha_Hora', ascending=True).reset_index(drop=True)
            df['N_Ejecucion'] = df.index + 1

            tab_gen, tab_vid, tab_img = st.tabs(["📊 General", "🎬 Detalle Video", "🖼️ Detalle Imagen"])

            with tab_gen:
                c1, c2, c3, c4 = st.columns(4) # Nueva columna para botón PDF
                c1.metric("Total Análisis", len(df))
                c2.metric("Modelo Favorito", df['Modelo'].mode()[0] if not df.empty else "-")
                c3.metric("CPU Media", f"{df['CPU_Usage'].mean():.1f}%")

                with c4:
                    st.write("") # Espaciador vertical
                    if st.button("🗑️ Borrar Historial"):
                        if os.path.exists(LOG_FILE): os.remove(LOG_FILE)
                        st.rerun()

                # --- NUEVO: BOTÓN DE DESCARGA PDF ---
                st.divider()
                if not df.empty:
                    pdf_bytes = create_pdf_report(df.sort_values('N_Ejecucion', ascending=False))
                    st.download_button(
                        label="📄 Descargar Informe PDF",
                        data=pdf_bytes,
                        file_name=f"Informe_SAR_{datetime.now().strftime('%Y%m%d')}.pdf",
                        mime="application/pdf",
                        help="Genera un informe técnico con el historial de ejecuciones."
                    )
                # ------------------------------------

                st.dataframe(df[['N_Ejecucion', 'Fecha', 'Hora', 'Tipo', 'Modelo', 'FPS_Promedio']].sort_values('N_Ejecucion', ascending=False), use_container_width=True)

            with tab_vid:
                df_v = df[df['Tipo'] == 'VIDEO']
                if not df_v.empty:
                    st.markdown("#### ⚡ Rendimiento FPS (Secuencial)")

                    fps_chart = alt.Chart(df_v).mark_line(point=True).encode(
                        x=alt.X('N_Ejecucion:Q', title='Nº de Análisis', axis=alt.Axis(tickMinStep=1)),
                        y=alt.Y('FPS_Promedio:Q', title='FPS'),
                        color=alt.Color('Modelo:N', legend=alt.Legend(title="Modelo IA", orient="top")),
                        tooltip=['N_Ejecucion', 'Fecha', 'Hora', 'Modelo', 'FPS_Promedio']
                    ).properties(height=400, title="Evolución de Rendimiento").interactive()

                    st.altair_chart(fps_chart, use_container_width=True)

                    st.divider()
                    c_a, c_b = st.columns(2)

                    with c_a:
                        chart_time = alt.Chart(df_v).mark_bar().encode(
                            x=alt.X('N_Ejecucion:Q', title='Nº Análisis', axis=alt.Axis(tickMinStep=1)),
                            y=alt.Y('Duracion_Sec:Q', title='Tiempo (s)'),
                            color='Modelo:N',
                            tooltip=['N_Ejecucion', 'Modelo', 'Duracion_Sec', 'Fecha']
                        ).properties(title="Tiempo de Análisis").interactive()
                        st.altair_chart(chart_time, use_container_width=True)

                    with c_b:
                        chart_cpu = alt.Chart(df_v).mark_area(opacity=0.5).encode(
                            x=alt.X('N_Ejecucion:Q', title='Nº Análisis', axis=alt.Axis(tickMinStep=1)),
                            y=alt.Y('CPU_Usage:Q', title='CPU %'),
                            color='Modelo:N',
                            tooltip=['N_Ejecucion', 'Modelo', 'CPU_Usage', 'Fecha']
                        ).properties(title="Consumo CPU").interactive()
                        st.altair_chart(chart_cpu, use_container_width=True)

            with tab_img:
                df_i = df[df['Tipo'] == 'IMAGEN']
                if not df_i.empty:
                    st.markdown("#### ⏱️ Latencia Imagen")
                    st.altair_chart(alt.Chart(df_i).mark_boxplot().encode(x='Modelo:N', y='Duracion_Sec:Q', color='Modelo:N').properties(height=400).interactive(), use_container_width=True)

        except Exception as e: st.error(f"Error procesando estadísticas: {e}")

In [ ]:
import os
import time
import subprocess
import re
import sys

# --- PASO 1: LIMPIEZA DE TERRENO ---
print(" 1. Limpiando procesos antiguos...")
# Matamos cualquier instancia previa de streamlit o el túnel
os.system("pkill -9 -f streamlit")
os.system("pkill -9 -f cloudflared")

# Borramos el historial de logs anterior para no confundir enlaces viejos
if os.path.exists("nohup.out"):
    os.remove("nohup.out")

# --- PASO 2: VERIFICACIÓN DE HERRAMIENTAS ---
print(" 2. Verificando dependencias...")
# Instalación silenciosa de librerías (solo si hacen falta)
subprocess.run("pip install -q streamlit opencv-python-headless ultralytics streamlit-option-menu", shell=True, check=False)

# Descarga del túnel si no existe
if not os.path.exists("cloudflared-linux-amd64"):
    print("   Descargando Cloudflare Tunnel...")
    subprocess.run("wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", shell=True)
    subprocess.run("chmod +x cloudflared-linux-amd64", shell=True)

# --- PASO 3: LANZAMIENTO ---
print(" 3. Iniciando Servidor SAR...")

# Verificamos que app.py exista antes de lanzar
if not os.path.exists("app.py"):
    print(" ERROR CRÍTICO: No se encuentra 'app.py'.")
    print("   Por favor, ejecuta primero la celda que crea el archivo app.py")
    sys.exit()

# Lanzamos Streamlit en segundo plano y enviamos la salida a nohup.out
os.system("nohup streamlit run app.py --server.port 8501 --server.headless true > nohup.out 2>&1 &")

# Damos un respiro para que arranque
time.sleep(3)

# Lanzamos el Túnel
print(" 4. Abriendo túnel seguro al mundo...")
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > nohup.out 2>&1 &")

# --- PASO 4: BÚSQUEDA INTELIGENTE DEL ENLACE ---
print(" Esperando confirmación de satélite (esto toma unos 15s)...")

link_encontrado = None
intentos = 0
max_intentos = 20 # 20 * 2s = 40 segundos máximo

while intentos < max_intentos:
    if os.path.exists("nohup.out"):
        with open("nohup.out", "r") as f:
            contenido = f.read()
            # Usamos Regex para capturar el enlace exacto
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', contenido)
            if match:
                link_encontrado = match.group(0)
                break

    time.sleep(2)
    intentos += 1
    # Feedback visual para que sepas que no se ha colgado
    if intentos % 5 == 0:
        print(f"   ...sintonizando ({intentos}/{max_intentos})")

# --- RESULTADO FINAL ---
if link_encontrado:
    print("\n" + "" * 15)
    print("   SISTEMA EN LÍNEA - ACCESO CONCEDIDO")
    print("" * 15)
    print(f"\n {link_encontrado}  \n")
    print(" Nota: Si se queda cargando, recarga la página web.")
else:
    print("\n TIEMPO AGOTADO: No se pudo obtener el enlace.")
    print("--- Últimos logs del sistema para diagnóstico ---")
    os.system("tail -n 10 nohup.out")

# **Solucion de problemas**

In [ ]:
import os
import time

print(" DETENIENDO TODO...")
# Matamos procesos con fuerza bruta
os.system("pkill -9 -f streamlit")
os.system("pkill -9 -f cloudflared")
time.sleep(2)

# Borramos el archivo de log para evitar confusiones con links viejos
if os.path.exists("nohup.out"):
    os.remove("nohup.out")

print(" LIMPIEZA COMPLETADA. INICIANDO DE NUEVO...")

# 1. Arrancar Streamlit (La App)
# Usamos un puerto específico para asegurar que no choque
!streamlit run app.py --server.port 8501 &>/dev/null &

# 2. Esperar un poco a que la App cargue
time.sleep(4)

# 3. Arrancar el Túnel (El puente a internet)
print(" Lanzando el túnel a internet...")
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

# 4. Esperar y mostrar el link DEFINITIVO
print(" Esperando a que Cloudflare nos dé la dirección (5 seg)...")
time.sleep(8)

print("\nEnlace a la web: ")
!grep -o 'https://.*\.trycloudflare.com' nohup.out | tail -n 1